In [ ]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from medmnist import PneumoniaMNIST
from sklearn.preprocessing import MinMaxScaler

data_flag = 'pneumoniamnist'
download = True
train_dataset = PneumoniaMNIST(split='train', download=download)
val_dataset = PneumoniaMNIST(split='val', download=download)
test_dataset = PneumoniaMNIST(split='test', download=download)

def normalize_medmnist(data):
    scaler = MinMaxScaler()
    n_samples = data.shape[0]
    reshaped = data.reshape(n_samples, -1)
    normalized = scaler.fit_transform(reshaped)
    return normalized.reshape(n_samples, 1, 28, 28).astype(np.float32)

x_train = normalize_medmnist(train_dataset.imgs)
x_val = normalize_medmnist(val_dataset.imgs)
x_test = normalize_medmnist(test_dataset.imgs)

class BasePaperAutoencoder(nn.Module):
    def __init__(self):
        super(BasePaperAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 8, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(8, 8, kernel_size=3, stride=2, padding=1),
            nn.Flatten()
        )
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (8, 4, 4)),
            nn.Conv2d(8, 8, kernel_size=3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(8, 8, kernel_size=3, padding=1), nn.ReLU(),
            nn.Upsample(size=(28,28), mode='nearest'),
            nn.Conv2d(8, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

model = BasePaperAutoencoder()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_loader = DataLoader(torch.from_numpy(x_train), batch_size=32, shuffle=True)

for epoch in range(5):
    for data in train_loader:
        _, decoded = model(data)
        loss = criterion(decoded, data)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def extract_and_save(data, labels, filename):
    model.eval()
    with torch.no_grad():
        features, _ = model(torch.from_numpy(data))

    df = pd.DataFrame(features.numpy())
    df['label'] = labels.flatten()
    df.to_csv(filename, index=False)
    print(f"Saved: {filename}")

extract_and_save(x_train, train_dataset.labels, 'train_features.csv')
extract_and_save(x_val, val_dataset.labels, 'val_features.csv')
extract_and_save(x_test, test_dataset.labels, 'test_features.csv')

100%|██████████| 4.17M/4.17M [00:05<00:00, 810kB/s]


Saved: train_features.csv
Saved: val_features.csv
Saved: test_features.csv
